# 02. Tiền xử lý và tạo đặc trưng

Notebook này:

1. Làm sạch dữ liệu thời tiết.
2. Đưa chuỗi về tần suất 10 phút.
3. Điền giá trị thiếu theo từng tập thời gian.
4. Xử lý ngoại lệ bằng IQR học từ tập train.
5. Tạo đặc trưng chu kỳ ngày, tuần và năm.
6. Chia train/validation/test theo tỷ lệ 70/15/15.
7. Chuẩn hóa đặc trưng bằng `StandardScaler` chỉ học trên tập train.
8. Lưu ba tập dữ liệu vào `data/processed/`.

Cột `rain_target_mm` giữ lượng mưa nguyên gốc theo đơn vị mm để làm nhãn.
Cột `rain` trong tập đầu ra là phiên bản đã chuẩn hóa, dùng như một đặc trưng
lịch sử tại thời điểm đầu vào.

## 1. Thư viện và cấu hình

In [ ]:
import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

CONFIG = {
    "input_file": "../data/raw/weather.csv",
    "processed_dir": "../data/processed",
    "target_col": "rain",
    "freq": "10min",
    "train_ratio": 0.70,
    "val_ratio": 0.15,
}

INPUT_PATH = Path(CONFIG["input_file"])
OUTPUT_DIR = Path(CONFIG["processed_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Không tìm thấy {INPUT_PATH}. Hãy đặt dữ liệu gốc vào data/raw/."
    )

## 2. Đọc và làm sạch dữ liệu

In [ ]:
raw_df = pd.read_csv(INPUT_PATH)
raw_df.columns = [str(col).strip() for col in raw_df.columns]

date_candidates = ["date", "datetime", "date time", "timestamp", "time"]
lower_map = {col.lower(): col for col in raw_df.columns}
date_col = next(
    (lower_map[name] for name in date_candidates if name in lower_map),
    raw_df.columns[0],
)

parsed = pd.to_datetime(raw_df[date_col], errors="coerce")
if parsed.notna().mean() < 0.80:
    parsed = pd.to_datetime(raw_df[date_col], errors="coerce", dayfirst=True)
raw_df[date_col] = parsed

raw_df = (
    raw_df.dropna(subset=[date_col])
    .sort_values(date_col)
    .drop_duplicates(subset=[date_col], keep="last")
)

for col in raw_df.columns:
    if col != date_col:
        raw_df[col] = pd.to_numeric(raw_df[col], errors="coerce")

raw_df = raw_df.replace([np.inf, -np.inf], np.nan)
empty_cols = [
    col for col in raw_df.columns
    if col != date_col and raw_df[col].notna().sum() == 0
]
raw_df = raw_df.drop(columns=empty_cols)

target_matches = [
    col for col in raw_df.columns
    if col.lower() == CONFIG["target_col"].lower()
]
if not target_matches:
    raise KeyError(
        f"Không tìm thấy target '{CONFIG['target_col']}'. "
        f"Các cột hiện có: {list(raw_df.columns)}"
    )
TARGET_COL = target_matches[0]

df = raw_df.set_index(date_col).resample(CONFIG["freq"]).asfreq()
print("Kích thước sau khi đưa về tần suất 10 phút:", df.shape)

## 3. Chia dữ liệu theo thời gian và xử lý giá trị thiếu

In [ ]:
n_total = len(df)
train_end = int(n_total * CONFIG["train_ratio"])
val_end = int(
    n_total * (CONFIG["train_ratio"] + CONFIG["val_ratio"])
)

def fill_block(block):
    return (
        block.copy()
        .interpolate(method="time", limit_direction="both")
        .ffill()
        .bfill()
    )

train_df = fill_block(df.iloc[:train_end])
val_df = fill_block(df.iloc[train_end:val_end])
test_df = fill_block(df.iloc[val_end:])

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)
print("Missing còn lại:", sum(x.isna().sum().sum() for x in [
    train_df, val_df, test_df
]))

## 4. Xử lý ngoại lệ bằng IQR của tập train

In [ ]:
sparse_cols = {"rain", "raining", "swdr", "par", "max. par"}
iqr_bounds = {}

for col in train_df.columns:
    if col.lower() in sparse_cols:
        continue
    q1, q3 = train_df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    if np.isfinite(iqr) and iqr > 0:
        iqr_bounds[col] = (
            float(q1 - 1.5 * iqr),
            float(q3 + 1.5 * iqr),
        )

def clip_block(block):
    block = block.copy()
    for col, (low, high) in iqr_bounds.items():
        block[col] = block[col].clip(low, high)
    return block

train_df = clip_block(train_df)
val_df = clip_block(val_df)
test_df = clip_block(test_df)

print("Số cột được xử lý ngoại lệ:", len(iqr_bounds))

## 5. Tạo đặc trưng thời gian và Fourier

In [ ]:
def add_calendar_features(block):
    block = block.copy()
    index = block.index
    minute = index.hour * 60 + index.minute
    weekday = index.dayofweek
    day = index.dayofyear

    block["hour"] = index.hour
    block["day_of_week"] = weekday
    block["month"] = index.month
    block["is_weekend"] = (weekday >= 5).astype(int)

    block["sin_day"] = np.sin(2 * np.pi * minute / 1440.0)
    block["cos_day"] = np.cos(2 * np.pi * minute / 1440.0)
    block["sin_week"] = np.sin(2 * np.pi * weekday / 7.0)
    block["cos_week"] = np.cos(2 * np.pi * weekday / 7.0)
    block["sin_year"] = np.sin(2 * np.pi * day / 365.25)
    block["cos_year"] = np.cos(2 * np.pi * day / 365.25)
    return block

train_df = add_calendar_features(train_df)
val_df = add_calendar_features(val_df)
test_df = add_calendar_features(test_df)

FEATURE_COLS = list(train_df.columns)
print("Số đặc trưng trước chuẩn hóa:", len(FEATURE_COLS))

## 6. Chuẩn hóa và lưu dữ liệu

In [ ]:
# Lưu dữ liệu sạch, chưa chuẩn hóa để phục vụ kiểm tra và báo cáo.
core_df = pd.concat([train_df, val_df, test_df])
core_df.index.name = "date"
core_df.reset_index().to_csv(
    OUTPUT_DIR / "core_preprocessed_weather.csv",
    index=False,
)

# StandardScaler chỉ được fit trên train để tránh rò rỉ dữ liệu.
scaler = StandardScaler()
scaler.fit(train_df[FEATURE_COLS])

def make_model_frame(block):
    scaled = pd.DataFrame(
        scaler.transform(block[FEATURE_COLS]),
        index=block.index,
        columns=FEATURE_COLS,
    )
    # Nhãn giữ nguyên đơn vị mm; cột rain chuẩn hóa vẫn là đầu vào.
    scaled["rain_target_mm"] = block[TARGET_COL].to_numpy()
    scaled.index.name = "date"
    return scaled.reset_index()

final_train = make_model_frame(train_df)
final_val = make_model_frame(val_df)
final_test = make_model_frame(test_df)

final_train.to_csv(OUTPUT_DIR / "final_train.csv", index=False)
final_val.to_csv(OUTPUT_DIR / "final_val.csv", index=False)
final_test.to_csv(OUTPUT_DIR / "final_test.csv", index=False)

joblib.dump(scaler, OUTPUT_DIR / "standard_scaler.joblib")

metadata = {
    "date_col": "date",
    "target_col_raw": TARGET_COL,
    "target_col_model": "rain_target_mm",
    "feature_cols": FEATURE_COLS,
    "iqr_bounds": iqr_bounds,
    "config": CONFIG,
    "sizes": {
        "train": len(final_train),
        "validation": len(final_val),
        "test": len(final_test),
    },
}
with (OUTPUT_DIR / "preprocessing_metadata.json").open(
    "w", encoding="utf-8"
) as handle:
    json.dump(metadata, handle, ensure_ascii=False, indent=2)

print("Đã lưu:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print("-", path)

## 7. Kiểm tra dữ liệu đầu ra

In [ ]:
print("Train:", final_train.shape)
print("Validation:", final_val.shape)
print("Test:", final_test.shape)
display(final_train.head())

assert final_train.isna().sum().sum() == 0
assert final_val.isna().sum().sum() == 0
assert final_test.isna().sum().sum() == 0
assert final_train["date"].max() < final_val["date"].min()
assert final_val["date"].max() < final_test["date"].min()